# 05 — Model B: classifier trained on SRGAN images

Model B reuses Model A's MobileNetV2 binary architecture, ImageNet normalization, BCE-with-logits objective, optimizer policy, metrics, and source-level train/validation partition. Its inputs come only from `data/generated/model_b_train.csv`; the reserved test manifest is never loaded during training.

## 1. Local setup

Run this notebook from the local repository with the project virtual environment selected as the Jupyter kernel. It reads notebook 04's local generated images and writes resumable Model B checkpoints locally.

In [ ]:
import sys
from pathlib import Path

start_directory = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (start_directory, *start_directory.parents)
        if (candidate / 'pyproject.toml').is_file()
        and (candidate / 'src' / 'applied_ai_midterm').is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    codex_workspace = Path.home() / 'Documents' / 'Codex'
    if codex_workspace.is_dir():
        PROJECT_ROOT = next(
            (
                config_file.parent
                for config_file in codex_workspace.rglob('pyproject.toml')
                if config_file.parent.name == 'MLawson-Applied-AI-Midterm'
                and (config_file.parent / 'src' / 'applied_ai_midterm').is_dir()
            ),
            None,
        )
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Open this notebook from inside the MLawson-Applied-AI-Midterm repository.'
    )

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print(f'Local project root: {PROJECT_ROOT}')
print(f'Python kernel: {sys.executable}')

In [ ]:
import json
from dataclasses import asdict

import matplotlib.pyplot as plt
from torch import nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import StepLR

from applied_ai_midterm.classifier import create_binary_mobilenet
from applied_ai_midterm.config import load_config
from applied_ai_midterm.model_b import create_model_b_dataloaders
from applied_ai_midterm.training import (
    fit_classifier,
    seed_everything,
    select_device,
)

## 2. Paths and shared Model A policy

The canonical `train.csv` determines membership and the exact source-level validation partition. Model B's generated manifest and images are read from local `data/generated/`. Its checkpoints use a separate `model_b` directory and `classifier_b_*` filenames.

In [ ]:
config = load_config(PROJECT_ROOT / 'configs' / 'config.yaml')
seed_everything(config.random_seed)
device = select_device()

SOURCE_TRAIN_MANIFEST = PROJECT_ROOT / 'data' / 'splits' / 'train.csv'
GENERATED_MANIFEST = (
    PROJECT_ROOT / 'data' / 'generated' / 'model_b_train.csv'
)
CHECKPOINT_DIR = (
    PROJECT_ROOT / 'artifacts' / 'checkpoints' / 'model_b'
)
VALIDATION_RATIO = 0.20
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

print(f'Selected device: {device}')
if device.type == 'cpu':
    print('Warning: local training on CPU may be slow.')
print(f'Generated manifest: {GENERATED_MANIFEST}')
print(f'Model B checkpoints: {CHECKPOINT_DIR}')
if not SOURCE_TRAIN_MANIFEST.is_file():
    raise FileNotFoundError(
        f'Training manifest missing at {SOURCE_TRAIN_MANIFEST}. Run notebook 01 first.'
    )
if not GENERATED_MANIFEST.is_file():
    raise FileNotFoundError(
        f'Generated manifest missing at {GENERATED_MANIFEST}. '
        'Run notebook 04 to completion, then Run all again.'
    )

## 3. Validate generated records and reproduce Model A's partition

Loading stops if a generated source is absent from canonical `train.csv`, if any source is duplicated or missing, if labels disagree, or if generator provenance is unavailable. Generated forms of a source record therefore cannot cross between training and validation.

In [ ]:
loaders = create_model_b_dataloaders(
    GENERATED_MANIFEST,
    SOURCE_TRAIN_MANIFEST,
    image_size=config.high_resolution_size,
    batch_size=config.classifier_batch_size,
    validation_ratio=VALIDATION_RATIO,
    random_seed=config.random_seed,
    num_workers=config.num_workers,
)
print(f'Training examples: {loaders.train_size:,}')
print(f'Validation examples: {loaders.validation_size:,}')
print(f'Class mapping: {loaders.class_mapping}')
print(
    'Generator checkpoint: '
    f"{loaders.generator_provenance['checkpoint_identifier']}"
)

## 4. Initialize or resume the shared classifier

This is the same single-logit MobileNetV2 and training policy used by Model A. `BCEWithLogitsLoss` receives raw logits; sigmoid is applied only by the shared metric function. Resume is rejected if the saved generator provenance or run configuration differs.

In [ ]:
model = create_binary_mobilenet(pretrained=True)
criterion = nn.BCEWithLogitsLoss()
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scheduler = StepLR(optimizer, step_size=7, gamma=0.1)
latest_checkpoint = CHECKPOINT_DIR / 'classifier_b_latest.pt'
resume_from = latest_checkpoint if latest_checkpoint.is_file() else None
resume_description = str(resume_from) if resume_from else 'none — epoch 1'
print(f'Resume checkpoint: {resume_description}')

## 5. Train Model B

This cell runs the configured 20 local training epochs. The latest checkpoint supports resumption and the best checkpoint is selected by validation F1. Histories and both checkpoints persist separately from Model A.

In [ ]:
run_configuration = {
    **asdict(config),
    'model': 'classifier_b',
    'architecture': 'mobilenet_v2',
    'pretrained_weights': 'MobileNet_V2_Weights.DEFAULT',
    'validation_ratio': VALIDATION_RATIO,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'scheduler': 'StepLR(step_size=7, gamma=0.1)',
    'generator_provenance': loaders.generator_provenance,
}
history = fit_classifier(
    model,
    loaders.train,
    loaders.validation,
    optimizer,
    criterion,
    epochs=config.classifier_epochs,
    device=device,
    checkpoint_dir=CHECKPOINT_DIR,
    class_mapping=loaders.class_mapping,
    random_seed=config.random_seed,
    configuration=run_configuration,
    scheduler=scheduler,
    resume_from=resume_from,
    checkpoint_prefix='classifier_b',
)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
history_path = CHECKPOINT_DIR / 'classifier_b_history.json'
history_path.write_text(json.dumps(history, indent=2), encoding='utf-8')
print(f'History saved: {history_path}')
print(f"Best checkpoint: {CHECKPOINT_DIR / 'classifier_b_best.pt'}")

## 6. Development curves

These curves use only the source-matched development split. Reserved-test comparison of Models A and B belongs in the next evaluation phase.

In [ ]:
train_history = history['train']
validation_history = history['validation']
epochs = [int(row['epoch']) for row in train_history]

figure, axes = plt.subplots(1, 3, figsize=(15, 4))
for split_name, rows in (
    ('Train', train_history),
    ('Validation', validation_history),
):
    axes[0].plot(epochs, [row['loss'] for row in rows], label=split_name)
    axes[1].plot(epochs, [row['f1'] for row in rows], label=split_name)
    axes[2].plot(epochs, [row['roc_auc'] for row in rows], label=split_name)
for axis, title in zip(axes, ('Loss', 'F1', 'ROC AUC'), strict=True):
    axis.set_title(title)
    axis.set_xlabel('Epoch')
    axis.legend()
    axis.grid(alpha=0.25)
plt.suptitle('Model B development history')
plt.tight_layout()

print('Final validation metrics:')
validation_history[-1]